# 9 - Local Admin and Maintenance

This notebook shows the operational side of SochDB's embedded model.

The goal is not to demonstrate every possible administration feature. The goal is to show a practical first admin story that a Python user can actually run:

- local embedded database directories
- basic stats and compression settings
- checkpoint and durable sync
- local backup creation and verification
- garbage collection and state inspection

That makes SochDB easier to understand as a real local system, not just a query demo.


### Step 0: Install Packages

Recommended install:

```bash
pip install sochdb
```

In [2]:
!pip install sochdb

import json
import shutil
import time
from pathlib import Path


### Step 1: Initialize a Local Database

We create a local embedded database directory and populate it with a small amount of application state.

In [3]:
from sochdb import Database

DB_PATH = Path("./admin_demo_db")
BACKUP_DIR = Path("./admin_demo_backups")

for path in [DB_PATH, BACKUP_DIR]:
    if path.exists():
        shutil.rmtree(path)

db = Database.open(str(DB_PATH))

for i in range(20):
    payload = {
        "id": i,
        "value": f"data_{i}",
        "timestamp": int(time.time()),
        "kind": "record" if i % 2 == 0 else "event",
    }
    db.put(f"records/{i:04d}".encode(), json.dumps(payload).encode())

db.put(b"config/app_name", b"sochdb-admin-demo")
db.put(b"config/version", b"1.0.0")

print(f"Database path: {DB_PATH.resolve()}")
initial_records = list(db.scan(b'records/'))
print(f"Initial record count: {len(initial_records)}")


Database path: /Users/saisandeepkantareddy/Downloads/sochdb-workspace/sochdb-notebooks/admin_demo_db
Initial record count: 20


/var/folders/h_/6ssf9cyj3f18ljd2pg8k4dqh0000gn/T/ipykernel_64971/1687859757.py:25: DeprecationWarning: scan() is deprecated for prefix queries. Use scan_prefix() instead. scan() may return keys beyond the intended prefix, causing data leakage.
  initial_records = list(db.scan(b'records/'))


### Step 2: Inspect the Database Layout

Because SochDB is embedded, the database is just a directory on disk. That makes local administration much easier to reason about.

In [4]:
sorted(str(path.relative_to(DB_PATH)) for path in DB_PATH.rglob("*"))[:20]


['.lock', 'wal.log']

### Step 3: Inspect Stats and Run Maintenance Operations

The public Python package exposes a useful first set of operational methods directly, including stats, compression settings, checkpoint, and fsync.


In [5]:
stats = db.stats()
print("Basic stats:")
print(json.dumps(stats, indent=2))

print(f"Current compression: {db.get_compression()}")
db.set_compression("lz4")
print(f"Compression after update: {db.get_compression()}")

checkpoint_lsn = db.checkpoint()
db.checkpoint_full()
db.fsync()
print(f"Checkpoint and fsync completed. Last checkpoint LSN: {checkpoint_lsn}")


Basic stats:
{
  "keys_count": 22,
  "memtable_size_bytes": 1721,
  "wal_size_bytes": 3965,
  "active_transactions": 0,
  "min_active_snapshot": 25,
  "last_checkpoint_lsn": 0
}
Current compression: none
Compression after update: lz4
Checkpoint and fsync completed. Last checkpoint LSN: 0


### Step 4: Create and Verify a Local Backup

Backups are part of the admin story people expect. Here we create one local backup artifact and verify that it is recognized by the package.


In [6]:
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
backup_path = BACKUP_DIR / "backup_001"
db.backup_create(str(backup_path))

print(f"Backup directory: {BACKUP_DIR.resolve()}")
print(f"Backup verify: {Database.backup_verify(str(backup_path))}")
print("Known backups:")
for backup in Database.backup_list(str(BACKUP_DIR)):
    print(backup)


Backup directory: /Users/saisandeepkantareddy/Downloads/sochdb-workspace/sochdb-notebooks/admin_demo_backups
Backup verify: True
Known backups:
{'name': 'sochdb-backup-1775519197869845', 'timestamp': 'sochdb-backup-1775519197869845', 'size_bytes': 0}


### Step 5: Modify State and Run Garbage Collection

Now we change the current state so we can inspect how the local database evolves during the session, then reclaim old MVCC versions.


In [7]:
for i in range(5):
    db.delete(f"records/{i:04d}".encode())

for i in range(20, 25):
    payload = {
        "id": i,
        "value": f"new_data_{i}",
        "timestamp": int(time.time()),
        "kind": "record",
    }
    db.put(f"records/{i:04d}".encode(), json.dumps(payload).encode())

primary_records = list(db.scan(b"records/"))
gc_count = db.gc()

print(f"Current record count after changes: {len(primary_records)}")
print(f"Garbage collection completed. Reclaimed {gc_count} old version(s).")


Current record count after changes: 20
Garbage collection completed. Reclaimed 5 old version(s).


/var/folders/h_/6ssf9cyj3f18ljd2pg8k4dqh0000gn/T/ipykernel_64971/3368300821.py:13: DeprecationWarning: scan() is deprecated for prefix queries. Use scan_prefix() instead. scan() may return keys beyond the intended prefix, causing data leakage.
  primary_records = list(db.scan(b"records/"))


### Step 6: Verify the Current Database State

The main thing we want to confirm is that the current in-session database state matches the operations we ran, and that the richer stats surface is available.


In [8]:
print(f"Current records: {len(primary_records)}")
print(f"First deleted record still present? {db.get(b'records/0000') is not None}")
print(f"Newest record present? {db.get(b'records/0024') is not None}")
print("Full stats snapshot:")
print(json.dumps(db.stats_full(), indent=2))


Current records: 20
First deleted record still present? False
Newest record present? True
Full stats snapshot:
{
  "memtable_size_bytes": 2246,
  "wal_size_bytes": 6008,
  "active_transactions": 0,
  "min_active_snapshot": 42,
  "last_checkpoint_lsn": 77,
  "transactions_started": 40,
  "transactions_committed": 40,
  "transactions_aborted": 0,
  "queries_executed": 0,
  "bytes_written": 2186,
  "bytes_read": 5171
}


### Step 7: Inspect the Database Directory

The embedded admin path stays tangible because the database itself is just a local directory.

In [ ]:
print(sorted(str(path.relative_to(DB_PATH)) for path in DB_PATH.rglob("*"))[:20])


### Cleanup

Close both databases cleanly when the walkthrough is done.

In [ ]:
db.close()
print("Database closed.")
